In [1]:
import torch as t
import pandas as pd
import sys
import os
import json
import itertools
import numpy as np
import yaml
sys.path.append("../")
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [72]:
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)
        
with open('../psychometric_tests/hexaco_100_eval.yaml', 'r') as file:
    hexaco_eval = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_questions.yaml', 'r') as file:
    hexaco_questions = yaml.safe_load(file)

In [3]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [13]:
def get_refusal_rate(answers):  
    answers = pd.Series(answers)
    refusal_answers = answers[answers.isin(['Do not wish to answer','Do not wish to answer.'])]
    return np.round(len(refusal_answers)/len(answers),3).item()

def get_mean_metrics(input_list):
    
    list_mean = np.round(np.mean(input_list), 3).item()
    list_std = np.round(np.std(input_list), 3).item()
    return list_mean, list_std

In [18]:
def get_refusal_rate_dict(data_dir):
    refusal_rate_dict = {}
    for filename in os.listdir(data_dir):
        if ".json" in filename and "hexaco" in filename:
            answers = read_json(os.path.join(data_dir,filename))
            for i, answer_key in enumerate(answers):
                if answer_key not in refusal_rate_dict.keys():
                    refusal_rate_dict[answer_key] = {}

                refusal_rate = get_refusal_rate(list(itertools.chain(*answers[answer_key]['answers'])))    
                refusal_rate_dict[answer_key][filename.split(".")[0]] = refusal_rate
    
    return refusal_rate_dict

In [19]:
data_dir = "case_study_data"

In [20]:
pd.DataFrame(get_refusal_rate_dict(data_dir)).T

,huggingface_hexaco_answers_gpt-4_1-mini
d5ab7ce4-7b65-446b-9271-491ad7edcacb,0.0
1f29ab95-8460-4c0a-aed1-c01d0455d8a2,0.0
4b382861-4343-449e-b30b-a3ac946760d4,0.0
f2e83cdd-28b3-42fe-9d17-31258d7e2b7f,0.0
d2db2a8a-9ae3-473e-b0c3-a3d429df400e,0.0
ee335dcf-1d34-44df-b53a-66c445ecadb7,0.0
a32089a1-1740-417b-87f2-266d3db53b61,0.0
de216cca-84ff-4724-806a-ef96d530b450,0.0


In [21]:
def calculate_hexaco_score(trait, subtrait, answers, likert_scale = generation_config['likert_scale']):
    answer_dict = {}
    answers = [answer if answer in likert_scale else "Do not wish to answer" for answer in answers]
    answers = pd.Series(answers)
    subtrait_dict = hexaco_eval[trait][subtrait]
    indices = [idx - 1 for idx in subtrait_dict['indices']]
    trait_answers = answers[indices]
    refused_answers = trait_answers[trait_answers.isin(['Do not wish to answer','Do not wish to answer.'])]
    non_refused_answers = trait_answers[~trait_answers.isin(['Do not wish to answer','Do not wish to answer.'])]
    answer_indices = [likert_scale.index(answer) for answer in non_refused_answers]
    true_answer_indices = [6-idx if reverse else idx for idx,reverse in zip(answer_indices, subtrait_dict['reverse'])]
    
    answer_dict['answer_indices'] = answer_indices
    answer_dict['true_answer_indices'] = true_answer_indices
    answer_dict['n_answered_questions'] = len(true_answer_indices)
    answer_dict['n_refused_questions'] = len(refused_answers)
    answer_dict['trait'] = trait
    answer_dict['subtrait'] = subtrait
    answer_dict['subtrait_score'] = np.round(np.mean(true_answer_indices).item(),3)
    return answer_dict

def get_trait_scores(iteration, persona,answer, likert_scale,filename):
    subtrait_hexaco_scores = []
    trait_hexaco_scores = []
    for trait in hexaco_eval.keys():
        trait_hexaco_score = {}
        trait_hexaco_score['persona'] = persona
        trait_hexaco_score['iteration'] = iteration
        trait_hexaco_score['trait'] = trait
        trait_hexaco_score['true_answer_indices'] = []
        trait_hexaco_score['n_answered_questions'] = 0
        trait_hexaco_score['n_refused_questions'] = 0
        for subtrait in hexaco_eval[trait].keys():
            subtrait_hexaco_score = calculate_hexaco_score(trait, subtrait, answer, likert_scale)
            subtrait_hexaco_score['persona'] = persona
            subtrait_hexaco_score['iteration'] = iteration
            trait_hexaco_score['true_answer_indices'].extend(subtrait_hexaco_score['true_answer_indices'])
            trait_hexaco_score['n_answered_questions'] += subtrait_hexaco_score['n_answered_questions']
            trait_hexaco_score['n_refused_questions'] += subtrait_hexaco_score['n_refused_questions']
            if "inverted" in filename:
                subtrait_hexaco_score['likert_scale'] = "inverse"
            else:
                subtrait_hexaco_score['likert_scale'] = "normal"
            if "paraphrase" in filename:
                subtrait_hexaco_score['paraphrase'] = "paraphrase"
            else:
                subtrait_hexaco_score['paraphrase'] = "normal"
            if "without_no" in filename:
                subtrait_hexaco_score['refusal_allowed'] = "No Refusal"
            else:
                subtrait_hexaco_score['refusal_allowed'] = "Refusal"
            if "llama" in filename:
                subtrait_hexaco_score['model'] = "llama_3.2_1b_it"
            else:
                subtrait_hexaco_score['model'] = "gpt_4.1_mini"
                
            subtrait_hexaco_scores.append(subtrait_hexaco_score)
        trait_hexaco_score['trait_score'] = np.round(np.mean(trait_hexaco_score['true_answer_indices']).item(),3)
        if "inverted" in filename:
            trait_hexaco_score['likert_scale'] = "inverse"
        else:
            trait_hexaco_score['likert_scale'] = "normal"
        if "paraphrase" in filename:
            trait_hexaco_score['paraphrase'] = "paraphrase"
        else:
            trait_hexaco_score['paraphrase'] = "normal"
        if "without_no" in filename:
            trait_hexaco_score['refusal_allowed'] = "No Refusal"
        else:
            trait_hexaco_score['refusal_allowed'] = "Refusal"
        if "llama" in filename:
            trait_hexaco_score['model'] = "llama_3.2_1b_it"
        else:
            trait_hexaco_score['model'] = "gpt_4.1_mini"
        trait_hexaco_scores.append(trait_hexaco_score)
        
    return pd.DataFrame(trait_hexaco_scores), pd.DataFrame(subtrait_hexaco_scores)

In [24]:
def get_all_stats(data_dir):
    trait_df = pd.DataFrame()
    subtrait_df = pd.DataFrame()
    for filename in os.listdir(data_dir):
        if ".json" in filename and "hexaco" in filename:
            print(filename)
            persona_answers = read_json(os.path.join(data_dir,filename))
            for persona_key in persona_answers:
                answers = persona_answers[persona_key]
                for i,answer in enumerate(answers['answers']):
                    iteration_trait_df, iteration_subtrait_df = get_trait_scores(i,answers['config']['persona'],answer, generation_config['likert_scale'],filename)
                    subtrait_df = pd.concat([subtrait_df,iteration_subtrait_df], axis = 0)
                    trait_df = pd.concat([trait_df,iteration_trait_df], axis = 0)
    
    subtrait_df_grouped = subtrait_df.groupby(['trait','subtrait','persona','likert_scale','paraphrase','refusal_allowed','model'],as_index = False).\
    agg({"n_answered_questions":"sum","n_refused_questions":"sum","subtrait_score":["mean",np.std]})
    
    subtrait_df_grouped.columns = ['trait','subtrait','persona','likert_scale','paraphrase','refusal_allowed','model','n_answered_questions','n_refused_questions',"subtrait_score_mean","subtrait_score_std"]
    
    trait_df_grouped = trait_df.groupby(['trait','persona','likert_scale','paraphrase','refusal_allowed','model'],as_index = False).\
    agg({"n_answered_questions":"sum","n_refused_questions":"sum","trait_score":["mean",np.std]})
    
    trait_df_grouped.columns = ['trait','persona','likert_scale','paraphrase','refusal_allowed','model','n_answered_questions','n_refused_questions',"trait_score_mean","trait_score_std"]
    
    subtrait_df_grouped.loc[:,'refusal_rate'] = subtrait_df_grouped.loc[:,'n_refused_questions']/(subtrait_df_grouped.loc[:,'n_answered_questions'] + subtrait_df_grouped.loc[:,'n_refused_questions'])
    trait_df_grouped.loc[:,'refusal_rate'] = trait_df_grouped.loc[:,'n_refused_questions']/(trait_df_grouped.loc[:,'n_answered_questions'] + trait_df_grouped.loc[:,'n_refused_questions'])

    return subtrait_df_grouped, trait_df_grouped
    

In [25]:
subtrait_df_grouped, trait_df_grouped = get_all_stats(data_dir)

huggingface_hexaco_answers_gpt-4_1-mini.json


/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_30825/2207547720.py:16: FutureWarning: The provided callable <function std at 0x103e96de0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  agg({"n_answered_questions":"sum","n_refused_questions":"sum","subtrait_score":["mean",np.std]})
/var/folders/9b/k67tngbx13jgzw5kjx9_d0tc0000gn/T/ipykernel_30825/2207547720.py:21: FutureWarning: The provided callable <function std at 0x103e96de0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  agg({"n_answered_questions":"sum","n_refused_questions":"sum","trait_score":["mean",np.std]})


In [28]:
gpt_subtrait_stat_df = subtrait_df_grouped[subtrait_df_grouped['model'] == "gpt_4.1_mini"]
llama_subtrait_stat_df = subtrait_df_grouped[subtrait_df_grouped['model'] == "llama_3.2_1b_it"]

gpt_trait_stat_df = trait_df_grouped[trait_df_grouped['model'] == "gpt_4.1_mini"]
llama_trait_stat_df = trait_df_grouped[trait_df_grouped['model'] == "llama_3.2_1b_it"]

In [29]:
gpt_subtrait_stat_pivot_df = gpt_subtrait_stat_df.pivot(values = ['refusal_rate','subtrait_score_mean','subtrait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait','subtrait'])

gpt_trait_stat_pivot_df = gpt_trait_stat_df.pivot(values = ['refusal_rate','trait_score_mean','trait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait'])

In [30]:
llama_subtrait_stat_pivot_df = llama_subtrait_stat_df.pivot(values = ['refusal_rate','subtrait_score_mean','subtrait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait','subtrait'])

llama_trait_stat_pivot_df = llama_trait_stat_df.pivot(values = ['refusal_rate','trait_score_mean','trait_score_std',], columns=['paraphrase', 'likert_scale','refusal_allowed'], index=['persona','trait'])

In [43]:
gpt_trait_stat_pivot_df.columns = ['refusal_rate','trait_score_mean','trait_scor_std']
gpt_subtrait_stat_pivot_df.columns = ['refusal_rate','trait_score_mean','trait_scor_std']

In [47]:
gpt_subtrait_stat_pivot_df

refusal_rate  \
persona                              trait                  subtrait                               
1f29ab95-8460-4c0a-aed1-c01d0455d8a2 agreeableness          flexibility                      0.0   
                                                            forgiveness                      0.0   
                                                            gentleness                       0.0   
                                                            patience                         0.0   
                                     altruism               altruism                         0.0   
...                                                                                          ...   
f2e83cdd-28b3-42fe-9d17-31258d7e2b7f honest-humility        sincerity                        0.0   
                                     openness to experience aesthetic appreciation           0.0   
                                                            creativity                       0.0   
                                                            inquisitiveness                  0.0   
                                                            unconventionality                0.0   

                                                                                    trait_score_mean  \
persona                              trait                  subtrait                                   
1f29ab95-8460-4c0a-aed1-c01d0455d8a2 agreeableness          flexibility                         3.25   
                                                            forgiveness                         3.00   
                                                            gentleness                          3.25   
                                                            patience                            3.25   
                                     altruism               altruism                            4.00   
...                                                                                              ...   
f2e83cdd-28b3-42fe-9d17-31258d7e2b7f honest-humility        sincerity                           3.25   
                                     openness to experience aesthetic appreciation              3.50   
                                                            creativity                          3.25   
                                                            inquisitiveness                     3.25   
                                                            unconventionality                   3.50   

                                                                                    trait_scor_std  
persona                              trait                  subtrait                                
1f29ab95-8460-4c0a-aed1-c01d0455d8a2 agreeableness          flexibility                        NaN  
                                                            forgiveness                        NaN  
                                                            gentleness                         NaN  
                                                            patience                           NaN  
                                     altruism               altruism                           NaN  
...                                                                                            ...  
f2e83cdd-28b3-42fe-9d17-31258d7e2b7f honest-humility        sincerity                          NaN  
                                     openness to experience aesthetic appreciation             NaN  
                                                            creativity                         NaN  
                                                            inquisitiveness                    NaN  
                                                            unconventionality                  NaN  

[200 rows x 3 columns]

In [57]:
def traitdf_to_nested_dict(df: pd.DataFrame) -> dict:
    result = {}
    for persona, sub_df in df.groupby(level=0):
        result[persona] = sub_df.droplevel(0).to_dict(orient="index")
    return result

def subtraitdf_to_nested_dict(df: pd.DataFrame) -> dict:
    result = {}
    for persona, persona_df in df.groupby(level=0):
        trait_dict = {}
        for trait, trait_df in persona_df.groupby(level=1):
            trait_dict[trait] = trait_df.droplevel([0,1]).to_dict(orient="index")
        result[persona] = trait_dict
    return result

In [67]:
write_to_json(subtraitdf_to_nested_dict(gpt_subtrait_stat_pivot_df),
              "case_study_data/hexaco_subtrait_stat_summary.json")

In [64]:
write_to_json(traitdf_to_nested_dict(gpt_trait_stat_pivot_df), "case_study_data/hexaco_trait_stat_summary.json")

In [68]:
hexaco_answers =  read_json("case_study_data/huggingface_hexaco_answers_gpt-4_1-mini.json")

In [71]:
hexaco_answers['1f29ab95-8460-4c0a-aed1-c01d0455d8a2']

{'config': {'persona': '1f29ab95-8460-4c0a-aed1-c01d0455d8a2',
  'paraphrase': 'normal',
  'likert_scale': 'normal',
  'refusal_allowed': 'refusal',
  'model_name': 'gpt-4.1-mini'},
 'answers': [['Neutral',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Strongly Agree',
   'Neutral',
   'Agree',
   'Agree',
   'Disagree',
   'Agree',
   'Agree',
   'Agree',
   'Neutral',
   'Agree',
   'Neutral',
   'Disagree',
   'Disagree',
   'Neutral',
   'Agree',
   'Agree',
   'Agree',
   'Neutral',
   'Strongly Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Strongly Disagree',
   'Agree',
   'Disagree',
   'Agree',
   'Agree',
   'Agree',
   'Neutral',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Neutral',
   'Agree',
   'Agree',
   'Neutral',
   'Disagree',
   'Neutral',
   'Neutral',
   'Agree',
   'Agree',
   'Agree',
   'Agree',
   'Strongly Agree',
   'Agree',

In [73]:
persona_hexaco_answer_list = []
for persona_id in hexaco_answers.keys():
    answer = hexaco_answers[persona_id]
    
    persona_hexaco_answer_list.extend([{"persona_id":persona_id,
                                       "question": hexaco_questions[index],
                                       'answer': answer}for index, answer in enumerate(answer['answers'][0])])
        

In [77]:
pd.DataFrame(persona_hexaco_answer_list).to_csv("case_study_data/hexaco_question_answer.csv", index=False)

gpt_trait_stat_pivot_df.to_csv(os.path.join(data_dir,"gpt_trait_stat_pivot_df.csv"))
llama_trait_stat_pivot_df.to_csv(os.path.join(data_dir,"llama_trait_stat_pivot_df.csv"))
gpt_subtrait_stat_pivot_df.to_csv(os.path.join(data_dir,"gpt_subtrait_stat_pivot_df.csv"))
llama_subtrait_stat_pivot_df.to_csv(os.path.join(data_dir,"llama_subtrait_stat_pivot_df.csv"))